# Web Annotation Data Preparation Pipeline

This notebook prepares the human evaluation datasets for the web annotation platform:

1. **Task A (Detector / Threshold Calibration)**:
   - Stratified extraction of 240 samples from `val.csv` across all 10 customer service subtypes (15 samples for `legitimate_frustration_at_product` and 25 samples each for the remaining 9 subtypes).
   - Saved to: `data_processed/task_a/task_a_web_annotation_240.csv`.

2. **Task B (Detoxification Quality Evaluation)**:
   - Sampling 100 toxic-only test inputs balanced across difficulty bands (34 Hard, 33 Medium, 33 Easy) and 7 toxic subtypes, aligned across 3 generation models (SFT Baseline, Config C Phase 3, and Flat C) and 3 random seeds (42, 123, 777).
   - Saved to: `data_processed/task_b/task_b_raw_outputs_3systems_100samples.csv`.

3. **Task B Deduplication and Mapping**:
   - Deduplicating identical model outputs per input sentence to reduce annotation workload from 300 to 234 unique pairs.
   - Saved to: `data_processed/task_b/task_b_web_annotation_234_pairs.csv` and `data_processed/task_b/task_b_pair_mapping_300.csv`.


## 1. Load and Verify Source Validation Split (`val.csv`)


In [ ]:
import pandas as pd
from pathlib import Path

# Resolve path to validation split in 02_data_preparation
CSV_PATH = Path("../../../02_data_preparation/data/01_full_splits/val.csv")
if not CSV_PATH.exists():
    CSV_PATH = Path("02_data_preparation/data/01_full_splits/val.csv")
if not CSV_PATH.exists():
    CSV_PATH = Path("../01_full_splits/val.csv")

assert CSV_PATH.exists(), f"Source validation split not found: {CSV_PATH}"
df_val = pd.read_csv(CSV_PATH)

print(f"Validation dataset loaded successfully: {len(df_val):,} rows")
print(f"Columns: {list(df_val.columns)}")
display(df_val.head(2))


## 2. Subtype Distribution Analysis in Source Split


In [ ]:
# Compute distribution by subtype
subtype_stats = df_val["subtype"].value_counts().reset_index()
subtype_stats.columns = ["subtype", "count"]
subtype_stats["percentage"] = (subtype_stats["count"] / len(df_val) * 100).round(2)

print("=" * 20 + " SUBTYPE DISTRIBUTION IN VAL.CSV " + "=" * 20)
print(subtype_stats.to_string(index=False))


## 3. Stratified Sampling for Task A (n = 240)

**Sampling Strategy:**
- Minority subtype `legitimate_frustration_at_product`: takes all 15 available rows.
- Remaining 9 subtypes: 25 rows each ($9 \times 25 = 225$ rows).
- Total target sample size: $15 + 225 = 240$ rows.
- Minimum quota of up to 8 hard-band samples per subtype where available, with proportional allocation across remaining bands.
- Seed: 42 for reproducible selection.


In [ ]:
# =============================================================================
# STRATIFIED SAMPLING: 240 SAMPLES ACROSS 10 SUBTYPES
# =============================================================================
import numpy as np
import pandas as pd

subtypes = sorted(df_val["subtype"].unique())
sampling_targets = {st: (15 if st == "legitimate_frustration_at_product" else 25) for st in subtypes}

sampled_dfs = []
for st, n_target in sampling_targets.items():
    st_df = df_val[df_val["subtype"] == st]
    
    if st == "legitimate_frustration_at_product":
        sampled_dfs.append(st_df)
        continue
    
    # Minimum quota for hard band (up to 8 samples)
    available_hard = len(st_df[st_df["band"] == "hard"])
    n_hard = 8 if available_hard >= 8 else available_hard
    
    rem_target = n_target - n_hard
    non_hard_df = st_df[st_df["band"] != "hard"]
    non_hard_counts = non_hard_df["band"].value_counts()
    
    props = non_hard_counts / len(non_hard_df)
    raw_q = props * rem_target
    round_q = raw_q.round().astype(int).to_dict()
    
    diff = rem_target - sum(round_q.values())
    if diff != 0:
        top_band = non_hard_counts.index[0]
        round_q[top_band] += diff
        
    st_samples = []
    if n_hard > 0:
        st_samples.append(st_df[st_df["band"] == "hard"].sample(n=n_hard, random_state=42))
        
    for b, q in round_q.items():
        if q > 0:
            b_pool = st_df[st_df["band"] == b]
            st_samples.append(b_pool.sample(n=min(q, len(b_pool)), random_state=42))
            
    sampled_dfs.append(pd.concat(st_samples))

df_output1 = pd.concat(sampled_dfs, ignore_index=True).sample(frac=1.0, random_state=42).reset_index(drop=True)

print("=" * 20 + " TASK A SAMPLING RESULT " + "=" * 20)
print(f"Total sampled rows: {len(df_output1)} (Expected: 240)")
print("\n--- BAND DISTRIBUTION ---")
print(df_output1["band"].value_counts().to_string())
print("\n--- SUBTYPE DISTRIBUTION ---")
print(df_output1["subtype"].value_counts().to_string())


## 4. Export Task A Dataset (`task_a_web_annotation_240.csv`)


In [ ]:
# =============================================================================
# EXPORT TASK A DATASET (CLEAN SPECIAL TOKENS AND NORMALIZE WHITESPACE)
# =============================================================================
import re
from pathlib import Path

def clean_text(t):
    if not isinstance(t, str):
        return ""
    t = re.sub(r"</?s>", "", t)
    t = re.sub(r"<pad>", "", t)
    t = re.sub(r"<extra_id_\d+>", "", t)
    t = re.sub(r"\s+", " ", t).strip()
    return t

df_output1["text"] = df_output1["text"].apply(clean_text)

OUTPUT_DIR_A = Path("../task_a")
if not OUTPUT_DIR_A.exists():
    OUTPUT_DIR_A = Path("website/data_processed/task_a")
OUTPUT_DIR_A.mkdir(parents=True, exist_ok=True)

OUTPUT_FILE_A = OUTPUT_DIR_A / "task_a_web_annotation_240.csv"
df_output1.to_csv(OUTPUT_FILE_A, index=False, encoding="utf-8-sig")

print(f"Successfully saved Task A dataset to: {OUTPUT_FILE_A}")
print(f"File size: {OUTPUT_FILE_A.stat().st_size:,} bytes | Rows: {len(df_output1)}")


## 5. Independent Verification of Task A Export


In [ ]:
# Verify the exported Task A CSV from disk
check_path = OUTPUT_FILE_A if "OUTPUT_FILE_A" in locals() and OUTPUT_FILE_A.exists() else Path("../task_a/task_a_web_annotation_240.csv")
df_verify = pd.read_csv(check_path)

print(f"Verifying exported file: {check_path.name}")
print(f"Total rows read: {len(df_verify):,} (Expected: 240)")
assert len(df_verify) == 240, f"Error: Expected 240 rows, got {len(df_verify)}"
assert df_verify["subtype"].nunique() == 10, "Error: Missing subtypes in Task A export"
assert (df_verify["subtype"] == "legitimate_frustration_at_product").sum() == 15
print("Verification passed: Task A dataset is complete and correctly stratified.")


---
# PART 2: Task B Dataset Preparation (Multi-System Detoxification Evaluation)

**Design and Stratification Objectives:**
1. **Total Sample Size**: 100 toxic input sentences ($X$).
2. **100% Toxic Inputs**: Filtered strictly from the toxic-only test set (1,150 samples).
3. **Balanced Difficulty Bands**:
   - Hard: 34 instances (34%)
   - Medium: 33 instances (33%)
   - Easy: 33 instances (33%)
4. **Stratified Seeds**: Distributed across seeds 42 (34 instances), 123 (33 instances), 777 (33 instances).
5. **Aligned Model Outputs**: Parallel outputs from SFT Baseline, Config C (Phase 3 refinement), and Flat C.


In [ ]:
# =============================================================================
# 2.1. LOAD MODEL INFERENCE OUTPUTS AND IDENTIFY TOXIC-ONLY TEST SET (n = 1,150)
# =============================================================================
import pandas as pd
from pathlib import Path

RAW_DIR = Path("../../data_raw")
if not RAW_DIR.exists():
    RAW_DIR = Path("website/data_raw")

TEST_CSV_PATH = Path("../../../02_data_preparation/data/01_full_splits/test.csv")
if not TEST_CSV_PATH.exists():
    TEST_CSV_PATH = Path("02_data_preparation/data/01_full_splits/test.csv")

assert RAW_DIR.exists(), f"Raw directory not found: {RAW_DIR}"
assert TEST_CSV_PATH.exists(), f"Test split not found: {TEST_CSV_PATH}"

print("Loading model generation outputs from data_raw...")
df_sft = pd.read_csv(RAW_DIR / "SFT/baseline_per_sample.csv")
df_c42 = pd.read_csv(RAW_DIR / "ConfigC/per_sample_seed42_phase3_refinement.csv")
df_c123 = pd.read_csv(RAW_DIR / "ConfigC/per_sample_seed123_phase3_refinement.csv")
df_c777 = pd.read_csv(RAW_DIR / "ConfigC/per_sample_seed777_phase3_refinement.csv")
df_f42 = pd.read_csv(RAW_DIR / "ConfigCFlat/per_sample_seed42_phase_flat.csv")
df_f123 = pd.read_csv(RAW_DIR / "ConfigCFlat/per_sample_seed123_phase_flat.csv")
df_f777 = pd.read_csv(RAW_DIR / "ConfigCFlat/per_sample_seed777_phase_flat.csv")
df_test_full = pd.read_csv(TEST_CSV_PATH)

# Identify common input sentences across all 7 evaluation outputs
common_X_set = set.intersection(*[
    set(d["X"]) for d in [df_sft, df_c42, df_c123, df_c777, df_f42, df_f123, df_f777]
])
print(f"Total common input sentences across all 7 runs: {len(common_X_set):,}")

# Join label and subtype metadata from test.csv
df_common = df_sft[df_sft["X"].isin(common_X_set)][["X", "band"]].merge(
    df_test_full[["text", "label", "subtype"]], left_on="X", right_on="text"
).drop(columns=["text"])

print(f"Label distribution in common inputs: {df_common['label'].value_counts().to_dict()}")

# Filter toxic-only test subset (1,150 samples)
df_toxic_only = df_common[df_common["label"] == "toxic"].copy().reset_index(drop=True)
print("=" * 60)
print(f"Toxic-only test instances: {len(df_toxic_only):,} (Expected: 1,150)")
assert len(df_toxic_only) == 1150, f"Error: Toxic count is {len(df_toxic_only)}, expected 1,150"
print("=" * 60)

print("\n--- BAND DISTRIBUTION (1,150 SAMPLES) ---")
print(df_toxic_only["band"].value_counts().to_string())

print("\n--- SUBTYPE DISTRIBUTION (1,150 SAMPLES) ---")
print(df_toxic_only["subtype"].value_counts().to_string())


In [ ]:
# =============================================================================
# 2.2. SAMPLE 100 INSTANCES BALANCED ACROSS BAND AND SUBTYPE WITH BALANCED SEEDS
# =============================================================================
BAND_TARGETS = {"hard": 34, "medium": 33, "easy": 33}

sampled_bands = []
for band, b_target in BAND_TARGETS.items():
    b_df = df_toxic_only[df_toxic_only["band"] == band]
    b_subtypes = sorted(b_df["subtype"].unique())
    n_st = len(b_subtypes)
    base_q = b_target // n_st
    rem_q = b_target % n_st
    
    b_sampled = []
    for i, st in enumerate(b_subtypes):
        quota = base_q + (1 if i < rem_q else 0)
        st_pool = b_df[b_df["subtype"] == st]
        b_sampled.append(st_pool.sample(n=min(quota, len(st_pool)), random_state=42))
    
    merged_b = pd.concat(b_sampled)
    if len(merged_b) < b_target:
        diff = b_target - len(merged_b)
        rem_pool = b_df[~b_df.index.isin(merged_b.index)]
        merged_b = pd.concat([merged_b, rem_pool.sample(n=diff, random_state=42)])
    
    merged_b = merged_b.sort_values("subtype").reset_index(drop=True)
    
    # Stratified seed distribution:
    # Hard (34): 12 seed 42, 11 seed 123, 11 seed 777
    # Medium (33): 11 seed 42, 11 seed 123, 11 seed 777
    # Easy (33): 11 seed 42, 11 seed 123, 11 seed 777
    if band == "hard":
        seeds = [42, 123, 777] * 11 + [42]
    else:
        seeds = [42, 123, 777] * 11
    merged_b["seed"] = seeds
    
    sampled_bands.append(merged_b)

df_100_inputs = pd.concat(sampled_bands, ignore_index=True).sample(frac=1.0, random_state=42).reset_index(drop=True)

print("=" * 70)
print(f"Sampled {len(df_100_inputs)} toxic-only input sentences balanced across band and subtype.")
print("=" * 70)

print("\n--- SUBTYPE x BAND DISTRIBUTION ---")
print(pd.crosstab(df_100_inputs["subtype"], df_100_inputs["band"], margins=True).to_string())

print("\n--- BAND x SEED DISTRIBUTION ---")
print(pd.crosstab(df_100_inputs["band"], df_100_inputs["seed"], margins=True).to_string())


In [ ]:
# =============================================================================
# 2.3. ALIGN SYSTEM OUTPUTS (SFT, CONFIG C, FLAT C) AND EXPORT TASK B RAW DATASET
# =============================================================================
import re

def clean_text(t):
    if not isinstance(t, str):
        return ""
    t = re.sub(r"</?s>", "", t)
    t = re.sub(r"<pad>", "", t)
    t = re.sub(r"<extra_id_\d+>", "", t)
    t = re.sub(r"\s+", " ", t).strip()
    return t

map_sft = dict(zip(df_sft["X"], df_sft["Y"]))
map_c42 = dict(zip(df_c42["X"], df_c42["Y"]))
map_c123 = dict(zip(df_c123["X"], df_c123["Y"]))
map_c777 = dict(zip(df_c777["X"], df_c777["Y"]))
map_f42 = dict(zip(df_f42["X"], df_f42["Y"]))
map_f123 = dict(zip(df_f123["X"], df_f123["Y"]))
map_f777 = dict(zip(df_f777["X"], df_f777["Y"]))

df_final_100 = df_100_inputs.copy()
df_final_100["Y_SFT"] = df_final_100["X"].map(map_sft).apply(clean_text)

y_config_c = []
y_flat_c = []
for _, row in df_final_100.iterrows():
    x, sd = row["X"], row["seed"]
    yc = map_c42.get(x) if sd == 42 else (map_c123.get(x) if sd == 123 else map_c777.get(x))
    yf = map_f42.get(x) if sd == 42 else (map_f123.get(x) if sd == 123 else map_f777.get(x))
    y_config_c.append(clean_text(yc))
    y_flat_c.append(clean_text(yf))

df_final_100["Y_ConfigC"] = y_config_c
df_final_100["Y_FlatC"] = y_flat_c

df_final_100.insert(0, "stt", range(1, len(df_final_100) + 1))
cols = ["stt", "X", "Y_SFT", "Y_ConfigC", "Y_FlatC", "seed", "band", "subtype", "label"]
df_final_100 = df_final_100[cols]

OUTPUT_DIR_B = Path("../task_b")
if not OUTPUT_DIR_B.exists():
    OUTPUT_DIR_B = Path("website/data_processed/task_b")
OUTPUT_DIR_B.mkdir(parents=True, exist_ok=True)

out_path_b_raw = OUTPUT_DIR_B / "task_b_raw_outputs_3systems_100samples.csv"
df_final_100.to_csv(out_path_b_raw, index=False, encoding="utf-8-sig")

print(f"Task B raw output dataset saved to: {out_path_b_raw}")
print(f"Total rows: {len(df_final_100)}")
display(df_final_100.head(2))


In [ ]:
# =============================================================================
# 2.4. INDEPENDENT VERIFICATION OF TASK B RAW EXPORT
# =============================================================================
check_path_2 = out_path_b_raw if "out_path_b_raw" in locals() and out_path_b_raw.exists() else Path("../task_b/task_b_raw_outputs_3systems_100samples.csv")
assert check_path_2.exists(), f"Task B raw export not found: {check_path_2}"

df_check = pd.read_csv(check_path_2)

print(f"Verification report for: {check_path_2.name}")
print(f"Total rows: {len(df_check)} (Expected: 100)")
assert len(df_check) == 100, f"Error: Expected 100 rows, got {len(df_check)}"

# Verify stt indexing
assert "stt" in df_check.columns
assert (df_check["stt"] == list(range(1, 101))).all()

# Verify label purity
assert (df_check["label"] == "toxic").sum() == 100

# Verify no empty generations
for col in ["Y_SFT", "Y_ConfigC", "Y_FlatC"]:
    assert df_check[col].isnull().sum() == 0, f"Error: Null value in {col}"

# Verify band distribution
ct_bs = pd.crosstab(df_check["band"], df_check["seed"], margins=True)
print("\n--- BAND x SEED DISTRIBUTION ---")
print(ct_bs.to_string())
assert ct_bs.loc["hard", "All"] == 34
assert ct_bs.loc["medium", "All"] == 33
assert ct_bs.loc["easy", "All"] == 33

print("\nVerification passed: Task B raw dataset is complete and properly balanced.")


---
# PART 3: Post-Processing & Deduplication for Web Annotation (Task B)

**Rationale:**
Across the 3 evaluation systems (SFT, Config C, Flat C), models occasionally produce identical outputs for the same input sentence.
By deduplicating identical outputs into unique evaluation pairs:
- Reduces redundant human grading by 22% (from 300 to 234 pairs).
- Prevents artificial inflation of inter-annotator agreement metrics.
- Preserves full mapping back to each system via `task_b_pair_mapping_300.csv`.


In [ ]:
# =============================================================================
# POST-PROCESSING 1: NORMALIZE TEXT FOR TASK A EXPORT
# =============================================================================
import pandas as pd
import re
from pathlib import Path

thresh_file = Path("../task_a/task_a_web_annotation_240.csv")
if thresh_file.exists():
    df_thresh = pd.read_csv(thresh_file)
    df_thresh["text"] = df_thresh["text"].astype(str).str.strip()
    df_thresh.to_csv(thresh_file, index=False, encoding="utf-8-sig")
    print(f"Normalized Task A dataset: {thresh_file.name} ({len(df_thresh)} rows)")


In [ ]:
# =============================================================================
# POST-PROCESSING 2: VERIFY 'stt' INDEXING FOR TASK B RAW OUTPUTS
# =============================================================================
check_file = Path("../task_b/task_b_raw_outputs_3systems_100samples.csv")
if check_file.exists():
    df_c = pd.read_csv(check_file)
    if "stt" not in df_c.columns:
        df_c.insert(0, "stt", range(1, len(df_c) + 1))
        df_c.to_csv(check_file, index=False, encoding="utf-8-sig")
    print(f"Verified 'stt' sequence in: {check_file.name} ({len(df_c)} rows)")


### Text Cleaning and Unicode NFC Normalization


In [ ]:
# =============================================================================
# POST-PROCESSING 3: UNICODE NFC NORMALIZATION FOR ALL EVALUATION CSVs
# =============================================================================
import unicodedata
import pandas as pd
from pathlib import Path

target_files = [
    Path("../task_a/task_a_web_annotation_240.csv"),
    Path("../task_b/task_b_raw_outputs_3systems_100samples.csv"),
]

for fpath in target_files:
    if fpath.exists():
        df = pd.read_csv(fpath)
        for col in df.select_dtypes(include=object).columns:
            df[col] = df[col].apply(lambda x: unicodedata.normalize("NFC", str(x)).strip() if pd.notnull(x) else x)
        df.to_csv(fpath, index=False, encoding="utf-8-sig")
        print(f"Applied Unicode NFC normalization to: {fpath.name}")


### Duplication Analysis in Model Outputs (Task B: 234 Unique Pairs)


In [ ]:
# =============================================================================
# POST-PROCESSING 4: DUPLICATION ANALYSIS (TASK B: 234 UNIQUE PAIRS)
# =============================================================================
import pandas as pd
from pathlib import Path

raw_b_path = Path("../task_b/task_b_raw_outputs_3systems_100samples.csv")
assert raw_b_path.exists(), f"File not found: {raw_b_path}"

df_b = pd.read_csv(raw_b_path)
n_unique_list = []

for idx, row in df_b.iterrows():
    outs = {str(row["Y_SFT"]).strip(), str(row["Y_ConfigC"]).strip(), str(row["Y_FlatC"]).strip()}
    n_unique_list.append(len(outs))

df_b_stats = pd.Series(n_unique_list).value_counts().sort_index()
print("Distribution of unique output counts per input sentence:")
for k, v in df_b_stats.items():
    print(f"  {k} unique outputs: {v} inputs ({k * v} pairs)")
total_pairs = sum(k * v for k, v in df_b_stats.items())
print(f"Total unique evaluation pairs: {total_pairs} (Expected: 234)")


### Generate 234 Deduplicated Pairs and 300-Row Mapping Table

**Outputs:**
1. `task_b_web_annotation_234_pairs.csv`: 234 unique pairs shuffled with a distance constraint $\ge 10$ between identical `input_id` occurrences.
2. `task_b_pair_mapping_300.csv`: 300 model output mappings linking each system back to its respective `pair_id`.


In [ ]:
# =============================================================================
# POST-PROCESSING 5: BUILD 234 UNIQUE PAIRS AND 300-ROW MAPPING TABLE
# =============================================================================
import random
import pandas as pd
from pathlib import Path

TASK_B_DIR = Path("../task_b")
if not TASK_B_DIR.exists():
    TASK_B_DIR = Path("website/data_processed/task_b")

raw_b_path = TASK_B_DIR / "task_b_raw_outputs_3systems_100samples.csv"
assert raw_b_path.exists(), f"Source file not found: {raw_b_path}"

df_b = pd.read_csv(raw_b_path)

raw_pairs = []
mapping_rows = []
pair_counter = 1

for idx, row in df_b.iterrows():
    stt = int(row.get("stt", idx + 1))
    x_en = str(row["X"]).strip()
    band = str(row.get("band", ""))
    subtype = str(row.get("subtype", ""))
    label = str(row.get("label", ""))
    seed_val = int(row.get("seed", 42))

    sys_outputs = [
        ("SFT", str(row["Y_SFT"]).strip(), None),
        ("ConfigC", str(row["Y_ConfigC"]).strip(), seed_val),
        ("FlatC", str(row["Y_FlatC"]).strip(), seed_val),
    ]

    seen_outputs = {}
    for sys_name, out_en, s_val in sys_outputs:
        if out_en not in seen_outputs:
            pid = f"P{pair_counter:03d}"
            pair_counter += 1
            seen_outputs[out_en] = {
                "pair_id": pid,
                "systems": [],
                "system_records": []
            }
        seen_outputs[out_en]["systems"].append(sys_name)
        seen_outputs[out_en]["system_records"].append((sys_name, s_val))

    for out_en, data in seen_outputs.items():
        pid = data["pair_id"]
        raw_pairs.append({
            "pair_id": pid,
            "input_id": stt,
            "input_text": x_en,
            "output_text": out_en,
            "systems": "+".join(data["systems"]),
            "num_systems": len(data["systems"]),
            "band": band,
            "subtype": subtype,
            "original_label": label,
        })
        for sys_name, s_val in data["system_records"]:
            mapping_rows.append({
                "system_sample_id": f"T{stt:03d}-{sys_name}",
                "pair_id": pid,
                "input_id": stt,
                "system_id": sys_name,
                "seed": s_val,
                "output_text": out_en,
                "band": band,
                "subtype": subtype,
                "original_label": label,
            })

# Shuffle with minimum distance constraint between same input_id occurrences
rng = random.Random(42)
best_shuffled = None
for attempt in range(500):
    available = raw_pairs.copy()
    rng.shuffle(available)
    result = []
    failed = False
    for step in range(len(raw_pairs)):
        recent_ids = {it["input_id"] for it in result[-15:]}
        candidates = [it for it in available if it["input_id"] not in recent_ids]
        if not candidates:
            recent_ids = {it["input_id"] for it in result[-10:]}
            candidates = [it for it in available if it["input_id"] not in recent_ids]
        if not candidates:
            failed = True
            break
        chosen = rng.choice(candidates)
        available.remove(chosen)
        result.append(chosen)
    if not failed and len(result) == len(raw_pairs):
        best_shuffled = result
        break

for i, it in enumerate(best_shuffled):
    it["item_number"] = i + 1

df_234 = pd.DataFrame(best_shuffled)
df_mapping = pd.DataFrame(mapping_rows)

out_234_csv = TASK_B_DIR / "task_b_web_annotation_234_pairs.csv"
out_mapping_csv = TASK_B_DIR / "task_b_pair_mapping_300.csv"

df_234.to_csv(out_234_csv, index=False, encoding="utf-8-sig")
df_mapping.to_csv(out_mapping_csv, index=False, encoding="utf-8-sig")

print("=" * 70)
print("Successfully generated Task B web annotation assets:")
print(f"1. {out_234_csv.name} -> {len(df_234)} rows (234 unique evaluation pairs)")
print(f"2. {out_mapping_csv.name} -> {len(df_mapping)} rows (300 model output mappings)")
print("=" * 70)
